In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fair
from fair import FAIR
from fair.interface import fill, initialise
from fair.io import read_properties

print("FaIR version:", fair.__version__)  # needs >= 2.2 (fill_species_configs(file), override_defaults)

# Calibrated, observationally constrained parameter ensemble: fair-calibrate v1.4.1 (Smith et al. 2024, GMD).
# Source: https://github.com/OMS-NetZero/FAIR/tree/master/examples/data/calibrated_constrained_ensemble
REPO = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
CAL = REPO / "data" / "fair" / "calibrated_constrained_ensemble"
PARAMS_FILE = CAL / "calibrated_constrained_parameters_calibration1.4.1.csv"
SPECIES_FILE = CAL / "species_configs_properties_calibration1.4.1.csv"

df_configs = pd.read_csv(PARAMS_FILE, index_col=0)
print(len(df_configs), "ensemble members")

In [ ]:
f = FAIR(ch4_method="Thornhill2021")

In [ ]:
f.define_time(1750, 2100, 1)

scenarios = ["ssp126", "ssp245"]
f.define_scenarios(scenarios)
f.define_configs(df_configs.index)  # one config per calibrated ensemble member

In [ ]:
species, properties = read_properties(filename=SPECIES_FILE)  # species list matching the calibration
f.define_species(species, properties)
f.allocate()

In [ ]:
f.fill_from_rcmip()                    # SSP emissions, solar and volcanic forcing from RCMIP
f.fill_species_configs(SPECIES_FILE)   # species defaults that go with the calibration
f.override_defaults(PARAMS_FILE)       # calibrated climate + gas-cycle + forcing parameters, per member

# Solar and volcanic scale factors are part of the calibration. The FaIR docs apply them explicitly
# so that they are picked up; do the same here.
for sp in ("Volcanic", "Solar"):
    fill(f.forcing, f.forcing.sel(specie=sp) * df_configs[f"forcing_scale[{sp}]"].values.squeeze(), specie=sp)

initialise(f.concentration, f.species_configs["baseline_concentration"])
initialise(f.forcing, 0)
initialise(f.temperature, 0)
initialise(f.cumulative_emissions, 0)
initialise(f.airborne_emissions, 0)
initialise(f.ocean_heat_content_change, 0)

In [ ]:
f.run()
# f.temperature: (timebounds, scenario, config, layer); f.concentration / f.forcing: (timebounds, scenario, config, specie)

In [ ]:
years = f.timebounds


def band(ax, da, label):
    """Ensemble median with 5-95% range over the calibrated members."""
    lo, hi = da.quantile([0.05, 0.95], dim="config")
    ax.fill_between(years, lo, hi, alpha=0.25)
    ax.plot(years, da.median("config"), label=label)


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for scen in scenarios:
    tas = f.temperature.sel(scenario=scen, layer=0)
    tas = tas - tas.sel(timebounds=slice(1850, 1900)).mean("timebounds")  # anomaly vs 1850-1900
    band(axes[0], tas, scen)
    band(axes[1], f.concentration.sel(scenario=scen, specie="CO2"), scen)
    band(axes[2], f.forcing.sel(scenario=scen, specie="CO2"), scen)
axes[0].set(title="Surface temperature anomaly vs 1850-1900", ylabel="K")
axes[1].set(title="CO2 concentration", ylabel="ppm")
axes[2].set(title="CO2 effective radiative forcing", ylabel="W m$^{-2}$")
for ax in axes:
    ax.set_xlabel("year")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Climate sensitivity of the ensemble. ECS and TCR are not inputs in FaIR: they emerge from the
# energy-balance parameters (ECS = 0.5 * forcing_4co2 / ocean_heat_transfer[0]).
for name in ("ecs", "tcr"):
    q = f.ebms[name].quantile([0.05, 0.5, 0.95]).values
    print(f"{name.upper()}: median {q[1]:.2f} K, 5-95% {q[0]:.2f}-{q[2]:.2f} K")

In [13]:
"""
# Saving to netCDF and reloading later

f.temperature.to_netcdf("temperature.nc")
f.forcing.to_netcdf("forcing.nc")

import xarray as xr
temp = xr.open_dataarray("temperature.nc")

# Or in one file 

xr.Dataset({"temperature": f.temperature, "forcing": f.forcing}).to_netcdf("fair_out.nc")
"""



'\n# Saving to netCDF and reloading later\n\nf.temperature.to_netcdf("temperature.nc")\nf.forcing.to_netcdf("forcing.nc")\n\nimport xarray as xr\ntemp = xr.open_dataarray("temperature.nc")\n\n# Or in one file \n\nxr.Dataset({"temperature": f.temperature, "forcing": f.forcing}).to_netcdf("fair_out.nc")\n'

In [ ]:
# abrupt-4xCO2 vs piControl, concentration-driven, using the calibrated climate parameters.
# Only CO2, CH4 and N2O are defined; they are held at baseline except CO2, which is quadrupled at t=0.
f4 = FAIR()
f4.define_time(0, 150, 1)
f4.define_scenarios(["abrupt-4xCO2", "piControl"])
f4.define_configs(df_configs.index)

ghg = ["CO2", "CH4", "N2O"]
_, cal_props = read_properties(filename=SPECIES_FILE)
f4.define_species(ghg, {s: {**cal_props[s], "input_mode": "concentration"} for s in ghg})
f4.allocate()
f4.fill_species_configs(SPECIES_FILE)

# calibrated parameters per member: energy balance AND the CO2 forcing scale, so the forcing itself
# varies across members as in the calibration (columns for species not defined here are skipped)
f4.override_defaults(PARAMS_FILE)
fill(f4.climate_configs["stochastic_run"], False)  # deterministic: no internal variability

base = {s: float(f4.species_configs["baseline_concentration"].sel(specie=s).isel(config=0)) for s in ghg}
for scen in ("piControl", "abrupt-4xCO2"):
    for s in ghg:
        fill(f4.concentration, base[s], scenario=scen, specie=s)
fill(f4.concentration, 4 * base["CO2"], scenario="abrupt-4xCO2", specie="CO2")

initialise(f4.forcing, 0)
initialise(f4.temperature, 0)
initialise(f4.cumulative_emissions, 0)
initialise(f4.airborne_emissions, 0)
initialise(f4.ocean_heat_content_change, 0)
f4.run()

In [ ]:
yrs4 = f4.timebounds
tas_4x = f4.temperature.sel(scenario="abrupt-4xCO2", layer=0) - f4.temperature.sel(scenario="piControl", layer=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, da, title, unit in (
    (axes[0], tas_4x, "Surface temperature response to abrupt 4xCO2", "K"),
    (axes[1], f4.concentration.sel(scenario="abrupt-4xCO2", specie="CO2"), "CO2 concentration", "ppm"),
    (axes[2], f4.forcing.sel(scenario="abrupt-4xCO2", specie="CO2"), "CO2 effective radiative forcing", "W m$^{-2}$"),
):
    lo, hi = da.quantile([0.05, 0.95], dim="config")
    ax.fill_between(yrs4, lo, hi, alpha=0.25)
    ax.plot(yrs4, da.median("config"))
    ax.set(title=title, xlabel="years after quadrupling", ylabel=unit)
plt.tight_layout()
plt.show()

# List of possible forcings: 

## Emitted species (you supply emissions)

### Greenhouse gases

CO2 from fossil fuels and industry (CO2 FFI) and from land use (CO2 AFOLU). Total CO2 is diagnosed from the two.
CH4 and N2O.
Halogenated gases (f-gas type):
CFCs: CFC-11, -12, -113, -114, -115.
HCFCs: HCFC-22, -141b, -142b.
Other ozone-depleting gases: CCl4, CH3CCl3, CHCl3, CH2Cl2, CH3Cl, CH3Br, and the halons (1211, 1301, 2402, and possibly 1202).
HFCs: HFC-23, -32, -125, -134a, -143a, -152a, -227ea, -236fa, -245fa, -365mfc, -4310mee.
PFCs and related gases: CF4, C2F6, C3F8, c-C4F8, C4F10, C5F12, C6F14, C7F16, C8F18, SF6, NF3, SO2F2.

### Aerosol precursors

Sulfur (SO2), black carbon (BC), organic carbon (OC), NH3.

### Ozone precursors (short-lived)

NOx, VOC, CO. CH4 also feeds ozone.


## Forcing computed inside the model
These have no direct input. FaIR derives them from the emissions or concentrations above:

Aerosol-radiation interactions, from Sulfur, BC, OC and NH3.
Aerosol-cloud interactions, from Sulfur, BC and OC.
Tropospheric and stratospheric ozone, from NOx, VOC, CO, CH4, N2O and the halogenated gases.
Light-absorbing particles on snow and ice (BC on snow), from BC.
Land-use albedo change, from CO2 AFOLU.
Stratospheric water vapour, from CH4.
Contrails, from aviation NOx in the v2.0 formulation. Check how your version handles contrails.

## Forcing you prescribe

Solar and volcanic, which fill_from_rcmip() fills for you.
You can also set any species to input_mode = "forcing" and supply W m-2 directly.

## FaIR is a global-mean, zero-dimensional model --> no local studeis e.g. not useful for aerosols --> SST